# Why your boarding pass spells you MUELLER

The same name, three encodings: what the machine-readable zone of a passport
does, what happens when two programs disagree about bytes, and what UTF-8 fixed.

In [1]:
name = "Jörg Müller"
print(name.encode("utf-8"))      # ö is two bytes: C3 B6
print(name.encode("latin-1"))    # ö is one byte:  F6

b'J\xc3\xb6rg M\xc3\xbcller'
b'J\xf6rg M\xfcller'


## Mojibake: reading UTF-8 bytes as if they were Latin-1

In [2]:
raw = name.encode("utf-8")
print(raw.decode("latin-1"))     # every byte becomes one character

JÃ¶rg MÃ¼ller


Nothing crashed, because both results are valid strings. The bytes never
changed; the *assumption* about them did.

## ASCII cannot spell ö at all

In [3]:
try:
    name.encode("ascii")
except UnicodeEncodeError as e:
    print("ASCII refuses:", e)

ASCII refuses: 'ascii' codec can't encode character '\xf6' in position 1: ordinal not in range(128)


## The machine-readable zone

ICAO Doc 9303 allows 37 symbols (A–Z, 0–9 and `<`) so that any reader anywhere
decodes the line identically. German umlauts and ß have their own
transliterations (ö becomes OE, ß becomes SS); every other accented letter
loses its accent. The line is padded with `<` to 44 characters.

In [4]:
import unicodedata

TRANSLIT = {"Ä": "AE", "Ö": "OE", "Ü": "UE", "ä": "AE", "ö": "OE", "ü": "UE", "ß": "SS",
            "Æ": "AE", "æ": "AE", "Ø": "OE", "ø": "OE"}
MRZ_ALPHABET = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<")

def to_mrz(text):
    """Transliterate a name into the 37-symbol MRZ alphabet."""
    out = []
    for ch in text:
        if ch in TRANSLIT:                                   # the ICAO table for these letters
            out.append(TRANSLIT[ch])
        else:                                                # Ç is C plus a cedilla mark: drop the mark
            parts = unicodedata.normalize("NFKD", ch)
            out.append("".join(c for c in parts if not unicodedata.combining(c)).upper())
    result = "".join(out).replace(" ", "<")
    assert set(result) <= MRZ_ALPHABET, result
    return result

def mrz_line(surname, given):
    line = f"P<D<<{to_mrz(surname)}<<{to_mrz(given)}"
    return line.ljust(44, "<")[:44]

print(mrz_line("Müller", "Jörg"))
print(mrz_line("Çağla", "Ayşe"))

P<D<<MUELLER<<JOERG<<<<<<<<<<<<<<<<<<<<<<<<<
P<D<<CAGLA<<AYSE<<<<<<<<<<<<<<<<<<<<<<<<<<<<


**What we used:** everything stored is a number, and a text encoding is the
convention that maps numbers to characters (session 2, sections 3 and 9).
The practical rule: always say which encoding, as in `open(path, encoding="utf-8")`.